# 데이터 유형, 그래픽 마크 및 시각적 인코딩 채널

시각화는 일련의 *그래픽 마크(graphical marks)* (막대, 선, 점 등)를 사용하여 데이터를 나타냅니다. 마크의 위치, 모양, 크기 또는 색상과 같은 속성은 기본 데이터 값을 인코딩할 수 있는 *채널(channels)* 역할을 합니다.

*데이터 유형(data types)*, *마크*, *인코딩 채널*의 기본 프레임을 통해 다양한 시각화를 간결하게 만들 수 있습니다. 이 노트북에서는 이러한 각 요소를 살펴보고 이를 사용하여 사용자 정의 통계 그래픽을 만드는 방법을 보여줍니다.

_이 노트북은 [데이터 시각화 커리큘럼](https://github.com/uwdata/visualization-curriculum)의 일부입니다._

In [1]:
import pandas as pd
import altair as alt

## 글로벌 개발 데이터


우리는 1955년부터 2005년까지 여러 국가의 글로벌 보건 및 인구 데이터를 시각화해 보겠습니다. 이 데이터는 [Gapminder 재단](https://www.gapminder.org/)에서 수집했으며 [Hans Rosling의 인기 있는 TED 강연](https://www.youtube.com/watch?v=hVimVzgtD6w)에서 공유되었습니다. 이 강연을 보지 않으셨다면 먼저 시청하시기를 권장합니다!

대한민국(South Korea)은 이 데이터 세트가 다루는 기간 동안 매우 드라마틱한 발전을 이룬 국가 중 하나입니다. 먼저 [vega-datasets](https://github.com/vega/vega-datasets) 컬렉션에서 데이터 세트를 Pandas 데이터 프레임으로 로드해 보겠습니다.

In [2]:
from vega_datasets import data as vega_data
data = vega_data.gapminder()

데이터의 크기는 얼마인가요?

In [3]:
data.shape

(693, 6)

693행 6열이네요! 데이터 내용을 한번 살펴봅시다:

In [4]:
data.head(5)

,year,country,cluster,pop,life_expect,fertility
0,1955,Afghanistan,0,8891209,30.332,7.7
1,1960,Afghanistan,0,9829450,31.997,7.7
2,1965,Afghanistan,0,10997885,34.020,7.7
3,1970,Afghanistan,0,12430623,36.088,7.7
4,1975,Afghanistan,0,14132019,38.438,7.7


각 `국가(country)`와 `연도(year)`(5년 간격)에 대해 여성 1인당 자녀 수인 출산율(`fertility`), 기대 수명(`life_expect`), 총 인구(`pop`) 측정값이 있습니다.

또한 정수 코드가 포함된 `cluster` 필드도 보입니다. 이것은 무엇을 나타낼까요? 데이터를 시각화하면서 이 미스터리를 풀어보겠습니다!

또한 2000년 데이터만 필터링한 작은 데이터 프레임을 만들어 보겠습니다.

In [5]:
data2000 = data.loc[data['year'] == 2000]

In [6]:
data2000.head(5)

,year,country,cluster,pop,life_expect,fertility
9,2000,Afghanistan,0,23898198,42.129,7.4792
20,2000,Argentina,3,37497728,74.340,2.3500
31,2000,Aruba,3,69539,73.451,2.1240
42,2000,Australia,4,19164620,80.370,1.7560
53,2000,Austria,1,8113413,78.980,1.3820


## 데이터 유형 (Data Types)


효과적인 시각화의 첫 번째 요소는 입력 데이터입니다. 데이터 값은 서로 다른 형태의 측정을 나타낼 수 있습니다. 이러한 측정은 어떤 종류의 비교를 지원할까요? 그리고 어떤 종류의 시각적 인코딩이 이러한 비교를 지원할까요?

Altair가 시각적 인코딩 선택을 알리는 데 사용하는 기본 데이터 유형을 살펴보는 것부터 시작하겠습니다. 이러한 데이터 유형은 우리가 할 수 있는 비교의 종류를 결정하고, 그에 따라 시각화 디자인 결정을 내리는 가이드 역할을 합니다.

### 명목형 (Nominal, N)

*명목형* 데이터(또는 *범주형* 데이터)는 범주 이름으로 구성됩니다.

명목형 데이터를 사용하면 값의 동등성을 비교할 수 있습니다: *값 A가 값 B와 같은가 아니면 다른가? (A = B)*. "A는 B와 같다" 또는 "A는 B와 같지 않다"와 같은 진술을 지원합니다.
위의 데이터 세트에서 `country` 필드는 명목형입니다.

명목형 데이터를 시각화할 때 값이 같은지 다른지 쉽게 확인할 수 있어야 합니다: 위치, 색상 색조(파란색, 빨간색, 초록색 등) 및 모양이 도움이 될 수 있습니다. 그러나 명목형 데이터를 인코딩하는 데 크기 채널을 사용하면 존재하지 않는 값 간의 순위나 크기 차이를 암시하여 우리를 오도할 수 있습니다!

### 서수형 (Ordinal, O)

*서수형* 데이터는 특정 순서가 있는 값으로 구성됩니다.

서수형 데이터를 사용하면 값의 순위를 비교할 수 있습니다: *값 A가 값 B보다 앞선가 아니면 뒤에 오는가? (A < B)*. "A는 B보다 작다" 또는 "A는 B보다 크다"와 같은 진술을 지원합니다.
위의 데이터 세트에서 `year` 필드를 서수형으로 처리할 수 있습니다.

서수형 데이터를 시각화할 때는 순위 감각을 인지할 수 있어야 합니다. 위치, 크기 또는 색상 값(밝기)이 적절할 수 있는 반면, 지각적으로 순서가 없는 색상 색조는 덜 적절할 것입니다.

### 정량적 (Quantitative, Q)

*정량적* 데이터를 사용하면 값 사이의 수치적 차이를 측정할 수 있습니다. 정량적 데이터에는 여러 하위 유형이 있습니다:

*간격(interval)* 데이터의 경우 지점 간의 거리(간격)를 측정할 수 있습니다: *값 B에서 값 A까지의 거리는 얼마인가? (A - B)*. "A는 B에서 12단위 떨어져 있다"와 같은 진술을 지원합니다.

*비율(ratio)* 데이터의 경우 0점이 의미가 있으므로 비율이나 스케일 팩터도 측정할 수 있습니다: *값 A는 값 B의 어느 정도 비율인가? (A / B)*. "A는 B의 10%이다" 또는 "B는 A보다 7배 크다"와 같은 진술을 지원합니다.

위의 데이터 세트에서 `year`는 정량적 간격 필드(연도 "0"의 값은 주관적임)인 반면, `fertility` 및 `life_expect`는 정량적 비율 필드(비율 계산을 위해 0이 의미 있음)입니다.
Vega-Lite는 정량적 데이터를 나타내지만 간격 유형과 비율 유형을 구분하지 않습니다.

정량적 값은 다른 채널 중에서도 위치, 크기 또는 색상 값을 사용하여 시각화할 수 있습니다. 비율 값의 비례 비교에는 0 기준선이 있는 축이 필수적이지만 간격 비교의 경우 안전하게 생략할 수 있습니다.

### 시간형 (Temporal, T)

*시간형* 값은 시점 또는 시간 간격을 측정합니다. 이 유형은 풍부한 의미와 관습(즉, [그레고리력](https://en.wikipedia.org/wiki/Gregorian_calendar))을 가진 정량적 값(타임스탬프)의 특수한 경우입니다. Vega-Lite의 시간형 유형은 시간 단위(년, 월, 일, 시 등)에 대한 추론을 지원하고 특정 시간 간격을 요청하기 위한 메서드를 제공합니다.

예시 시간형 값에는 `“2019-01-04”` 및 `“Jan 04 2019”`와 같은 날짜 문자열뿐만 아니라 [ISO 날짜-시간 형식](https://en.wikipedia.org/wiki/ISO_8601)인 `“2019-01-04T17:50:35.643Z”`와 같은 표준화된 날짜-시간이 포함됩니다.

위의 글로벌 개발 데이터 세트에는 `year` 필드가 단순히 정수로 인코딩되어 있기 때문에 시간형 값이 없습니다. Altair에서 시간형 데이터를 사용하는 방법에 대한 자세한 내용은 [시간 및 날짜 문서](https://altair-viz.github.io/user_guide/times_and_dates.html)를 참조하세요.

### 요약

이러한 데이터 유형은 서로 배타적인 것이 아니라 계층을 형성합니다: 서수형 데이터는 명목형(동등성) 비교를 지원하고 정량적 데이터는 서수형(순위 순서) 비교를 지원합니다.

또한 이러한 데이터 유형은 고정된 범주화를 제공하지 않습니다. 데이터 필드가 숫자를 사용하여 표현된다고 해서 반드시 정량적 유형으로 처리해야 하는 것은 아닙니다! 예를 들어, 일련의 연령(10세, 20세 등)을 명목형(미성년자 또는 성인), 서수형(연령대별 그룹화) 또는 정량적(평균 연령 계산)으로 해석할 수 있습니다.

이제 이러한 데이터 유형을 시각적으로 인코딩하는 방법을 살펴보겠습니다!


## 인코딩 채널 (Encoding Channels)

Altair의 핵심은 데이터 필드(주어진 데이터 유형 포함)를 선택한 *마크* 유형의 사용 가능한 인코딩 *채널*에 바인딩하는 *인코딩*을 사용하는 것입니다. 이 노트북에서는 다음 인코딩 채널을 살펴보겠습니다:

- `x`: 마크의 수평(x축) 위치.
- `y`: 마크의 수직(y축) 위치.
- `size`: 마크의 크기. 마크 유형에 따라 면적 또는 길이에 해당할 수 있습니다.
- `color`: [유효한 CSS 색상](https://developer.mozilla.org/en-US/docs/Web/CSS/color_value)으로 지정된 마크 색상.
- `opacity`: 마크 불투명도, 0(완전 투명)에서 1(완전 불투명) 사이의 범위.
- `shape`: `point` 마크의 플로팅 심볼 모양.
- `tooltip`: 마크 위에 마우스를 올렸을 때 표시할 툴팁 텍스트.
- `order`: 마크 순서, 선/영역 포인트 순서 및 그리기 순서를 결정합니다.
- `column`: 데이터를 수평으로 정렬된 하위 플롯으로 나눕니다.
- `row`: 데이터를 수직으로 정렬된 하위 플롯으로 나눕니다.

사용 가능한 채널의 전체 목록은 [Altair 인코딩 문서](https://altair-viz.github.io/user_guide/encodings/index.html)를 참조하세요.

### X

`x` 인코딩 채널은 마크의 수평 위치(x 좌표)를 설정합니다. 또한 축과 제목의 기본 선택이 자동으로 이루어집니다. 아래 차트에서 정량적 데이터 유형을 선택하면 연속적인 선형 축 스케일이 생성됩니다:

In [7]:
alt.Chart(data2000).mark_point().encode(
    alt.X('fertility:Q')
)

alt.Chart(...)

### Y

`y` 인코딩 채널은 마크의 수직 위치(y 좌표)를 설정합니다. 여기서는 서수형(`O`) 데이터 유형을 사용하여 `cluster` 필드를 추가했습니다. 그 결과 각 고유 값에 대해 기본 단계 크기를 가진 대역(band)이 포함된 이산 축이 생성됩니다:

In [8]:
alt.Chart(data2000).mark_point().encode(
    alt.X('fertility:Q'),
    alt.Y('cluster:O')
)

alt.Chart(...)

_`O`와 `Q` 필드 유형을 바꾸면 위의 차트에 어떤 일이 일어날까요?_

대신 `life_expect` 필드를 정량적(`Q`) 변수로 추가하면 두 축 모두에 선형 스케일이 있는 산점도가 생성됩니다:

In [9]:
alt.Chart(data2000).mark_point().encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q')
)

alt.Chart(...)

기본적으로 선형 정량적 스케일의 축에는 비율 값 데이터를 비교하기 위한 적절한 기준선을 보장하기 위해 0이 포함됩니다. 그러나 어떤 경우에는 0 기준선이 무의미하거나 간격 비교에 집중하고 싶을 수도 있습니다. 0의 자동 포함을 비활성화하려면 인코딩 `scale` 속성을 사용하여 스케일 매핑을 구성합니다:

In [10]:
alt.Chart(data2000).mark_point().encode(
    alt.X('fertility:Q', scale=alt.Scale(zero=False)),
    alt.Y('life_expect:Q', scale=alt.Scale(zero=False))
)

alt.Chart(...)

이제 축 스케일에 더 이상 기본적으로 0이 포함되지 않습니다. 축 도메인 끝점이 자동으로 5 또는 10의 배수와 같은 *좋은(nice)* 숫자에 맞춰지므로 약간의 여백(padding)은 여전히 남아 있습니다.

_위의 스케일 속성에 `nice=False`를 추가하면 어떻게 될까요?_

### 크기 (Size)

`size` 인코딩 채널은 마크의 크기나 범위를 설정합니다. 채널의 의미는 마크 유형에 따라 다를 수 있습니다. `point` 마크의 경우, `size` 채널은 플로팅 심볼의 픽셀 면적에 매핑되므로 점의 직경은 크기 값의 제곱근과 일치합니다.

인구(`pop`)를 `size` 채널에 인코딩하여 산점도를 보강해 보겠습니다. 그 결과 차트에는 이제 크기 값을 해석하기 위한 범례도 포함됩니다.

In [11]:
alt.Chart(data2000).mark_point().encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Size('pop:Q')
)

alt.Chart(...)

경우에 따라 기본 크기 범위에 만족하지 못할 수도 있습니다. 사용자 정의된 크기 범위를 제공하려면 `scale` 속성의 `range` 매개변수를 가장 작은 크기와 가장 큰 크기를 나타내는 배열로 설정하세요. 여기서는 크기 인코딩이 0 픽셀(0 값의 경우)에서 1,000 픽셀(스케일 도메인의 최대 값의 경우) 사이의 범위를 갖도록 업데이트합니다:

In [12]:
alt.Chart(data2000).mark_point().encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Size('pop:Q', scale=alt.Scale(range=[0,1000]))
)

alt.Chart(...)

### 색상 및 불투명도 (Color and Opacity)

`color` 인코딩 채널은 마크의 색상을 설정합니다. 색상 인코딩 스타일은 데이터 유형에 따라 크게 달라집니다. 명목형 데이터는 기본적으로 여러 색조의 질적 색상표를 사용하고, 서수형 및 정량적 데이터는 지각적으로 순서가 지정된 색상 그라데이션을 사용합니다.

여기서는 명목형(`N`) 데이터 유형과 `color` 채널을 사용하여 `cluster` 필드를 인코딩하여 각 클러스터 값에 대해 별개의 색조가 생성되도록 합니다. `cluster` 필드가 무엇을 나타내는지 추측할 수 있나요?


In [13]:
alt.Chart(data2000).mark_point().encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Size('pop:Q', scale=alt.Scale(range=[0,1000])),
    alt.Color('cluster:N')
)

alt.Chart(...)

채워진 모양을 선호한다면 `mark_point` 메서드에 `filled=True` 매개변수를 전달할 수 있습니다:

In [14]:
alt.Chart(data2000).mark_point(filled=True).encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Size('pop:Q', scale=alt.Scale(range=[0,1000])),
    alt.Color('cluster:N')
)

alt.Chart(...)

기본적으로 Altair는 오버플로팅(over-plotting)을 방지하기 위해 약간의 투명도를 사용합니다. `mark_*` 메서드에 기본값을 전달하거나 전용 인코딩 채널을 사용하여 불투명도를 더 조정할 수 있습니다.

여기서는 데이터 필드를 바인딩하는 대신 인코딩 채널에 상수 값을 제공하는 방법을 보여줍니다:

In [15]:
alt.Chart(data2000).mark_point(filled=True).encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Size('pop:Q', scale=alt.Scale(range=[0,1000])),
    alt.Color('cluster:N'),
    alt.OpacityValue(0.5)
)

alt.Chart(...)

### 모양 (Shape)

`shape` 인코딩 채널은 `point` 마크에서 사용되는 기하학적 모양을 설정합니다. 지금까지 보았던 다른 채널과 달리 `shape` 채널은 다른 마크 유형에서는 사용할 수 없습니다. 모양 인코딩 채널은 지각적 순위 및 크기 비교가 지원되지 않으므로 명목형 데이터에만 사용해야 합니다.

`cluster` 필드를 `color`뿐만 아니라 `shape`를 사용하여 인코딩해 보겠습니다. 동일한 기본 데이터 필드에 대해 여러 채널을 사용하는 것을 *중복 인코딩(redundant encoding)*이라고 합니다. 결과 차트는 색상과 모양 정보를 단일 심볼 범례로 결합합니다:

In [16]:
alt.Chart(data2000).mark_point(filled=True).encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Size('pop:Q', scale=alt.Scale(range=[0,1000])),
    alt.Color('cluster:N'),
    alt.OpacityValue(0.5),
    alt.Shape('cluster:N')
)

alt.Chart(...)

### 툴팁 및 정렬 (Tooltips & Ordering)

이쯤 되면 조금 답답할 수도 있습니다. 차트를 만들었지만 시각화된 점들이 어느 국가에 해당하는지 여전히 알 수 없기 때문입니다! 탐색을 가능하게 하기 위해 대화형 툴팁을 추가해 보겠습니다.

`tooltip` 인코딩 채널은 사용자가 마크 위로 마우스 커서를 이동할 때 표시할 툴팁 텍스트를 결정합니다. `country` 필드에 대한 툴팁 인코딩을 추가한 다음 어떤 국가가 표시되고 있는지 조사해 보겠습니다.


In [17]:
alt.Chart(data2000).mark_point(filled=True).encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Size('pop:Q', scale=alt.Scale(range=[0,1000])),
    alt.Color('cluster:N'),
    alt.OpacityValue(0.5),
    alt.Tooltip('country')
)

alt.Chart(...)

마우스를 움직이다 보면 일부 점을 선택할 수 없다는 것을 알 수 있습니다. 예를 들어, 가장 큰 짙은 파란색 원은 인도에 해당하는데, 인구수가 더 적은 국가 위에 그려져 있어 마우스가 그 국가 위로 올라가는 것을 방해합니다. 이 문제를 해결하기 위해 `order` 인코딩 채널을 사용할 수 있습니다.

`order` 인코딩 채널은 데이터 포인트의 순서를 결정하며, 그려지는 순서와 `line` 및 `area` 마크의 경우 서로 연결되는 순서 모두에 영향을 미칩니다.

인구(`pop`)를 기준으로 내림차순으로 값을 정렬하여 큰 원보다 작은 원이 나중에 그려지도록 해 보겠습니다:

In [18]:
alt.Chart(data2000).mark_point(filled=True).encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Size('pop:Q', scale=alt.Scale(range=[0,1000])),
    alt.Color('cluster:N'),
    alt.OpacityValue(0.5),
    alt.Tooltip('country:N'),
    alt.Order('pop:Q', sort='descending')
)

alt.Chart(...)

이제 인도에 가려져 있던 더 작은 국가를 식별할 수 있습니다. 바로 방글라데시입니다!

이제 `cluster` 필드가 무엇을 나타내는지도 파악할 수 있습니다. 다양한 색상의 점 위에 마우스를 올려 자신만의 설명을 만들어 보세요.

지금까지 기본 데이터 레코드의 단일 속성만 보여주는 툴팁을 추가했습니다. 여러 값을 표시하려면 `tooltip` 채널에 포함하려는 각 필드에 대해 하나씩 인코딩 배열을 제공할 수 있습니다:

In [19]:
alt.Chart(data2000).mark_point(filled=True).encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Size('pop:Q', scale=alt.Scale(range=[0,1000])),
    alt.Color('cluster:N'),
    alt.OpacityValue(0.5),
    alt.Order('pop:Q', sort='descending'),
    tooltip = [
        alt.Tooltip('country:N'),
        alt.Tooltip('fertility:Q'),
        alt.Tooltip('life_expect:Q')
    ]   
)

alt.Chart(...)

이제 마우스 호버 시 여러 데이터 필드를 볼 수 있습니다!

### 열 및 행 패싯 (Column and Row Facets)

공간 위치는 시각적 인코딩을 위한 가장 강력하고 유연한 채널 중 하나이지만, 이미 `x` 및 `y` 채널에 필드를 할당한 경우에는 무엇을 할 수 있을까요? 유용한 기술 중 하나는 데이터의 하위 세트를 보여주는 하위 플롯으로 구성된 *트렐리스 플롯(trellis plot)*을 만드는 것입니다. 트렐리스 플롯은 [작은 다중 창(small multiples)](https://en.wikipedia.org/wiki/Small_multiple) 뷰를 사용하여 데이터를 제공하는 보다 일반적인 기술의 한 예입니다.

`column` 및 `row` 인코딩 채널은 제공된 데이터 필드에 따라 데이터가 분할되는 수평(열) 또는 수직(행) 하위 플롯 세트를 생성합니다.

다음은 데이터를 `cluster` 값당 하나의 열로 나누는 트렐리스 플롯입니다:

In [20]:
alt.Chart(data2000).mark_point(filled=True).encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Size('pop:Q', scale=alt.Scale(range=[0,1000])),
    alt.Color('cluster:N'),
    alt.OpacityValue(0.5),
    alt.Tooltip('country:N'),
    alt.Order('pop:Q', sort='descending'),
    alt.Column('cluster:N')
)

alt.Chart(...)

위의 플롯은 화면에 다 들어오지 않아 모든 하위 플롯을 서로 비교하기 어렵습니다! 기본 `width` 및 `height` 속성을 설정하여 더 작은 다중 창 세트를 만들 수 있습니다. 또한 열 헤더에 이미 `cluster` 값이 레이블로 표시되어 있으므로 `color` 범례를 `None`으로 설정하여 제거하겠습니다. 공간을 더 잘 활용하기 위해 `size` 범례의 방향을 차트의 `'bottom'`으로 조정할 수도 있습니다.

In [21]:
alt.Chart(data2000).mark_point(filled=True).encode(
    alt.X('fertility:Q'),
    alt.Y('life_expect:Q'),
    alt.Size('pop:Q', scale=alt.Scale(range=[0,1000]),
             legend=alt.Legend(orient='bottom', titleOrient='left')),
    alt.Color('cluster:N', legend=None),
    alt.OpacityValue(0.5),
    alt.Tooltip('country:N'),
    alt.Order('pop:Q', sort='descending'),
    alt.Column('cluster:N')
).properties(width=135, height=135)

alt.Chart(...)

내부적으로 `column` 및 `row` 인코딩은 `facet` 뷰 구성 연산자를 사용하는 새 사양으로 변환됩니다. 나중에 패싯에 대해 더 자세히 알아볼 것입니다!

그동안 *위의 차트를 열 대신 행으로 패싯하도록 다시 작성할 수 있나요?*

### 미리 보기: 대화형 필터링 (Interactive Filtering)

나중에 나올 모듈에서는 데이터 탐색을 위한 상호작용 기술을 자세히 살펴볼 것입니다. 여기 미리 보기가 있습니다: `year` 필드에 범위 슬라이더를 바인딩하여 각 연도의 데이터를 대화식으로 훑어볼 수 있도록 합니다. 나중에 상호작용을 자세히 다룰 것이므로 아래 코드가 지금 시점에서 약간 혼란스럽더라도 걱정하지 마세요.

_슬라이더를 앞뒤로 드래그하여 시간이 지남에 따라 데이터 값이 어떻게 변하는지 확인해 보세요!_

In [22]:
select_year = alt.selection_single(
    name='select', fields=['year'], init={'year': 1955},
    bind=alt.binding_range(min=1955, max=2005, step=5)
)

alt.Chart(data).mark_point(filled=True).encode(
    alt.X('fertility:Q', scale=alt.Scale(domain=[0,9])),
    alt.Y('life_expect:Q', scale=alt.Scale(domain=[0,90])),
    alt.Size('pop:Q', scale=alt.Scale(domain=[0, 1200000000], range=[0,1000])),
    alt.Color('cluster:N', legend=None),
    alt.OpacityValue(0.5),
    alt.Tooltip('country:N'),
    alt.Order('pop:Q', sort='descending')
).add_selection(select_year).transform_filter(select_year)

alt.Chart(...)

## 그래픽 마크 (Graphical Marks)

위의 인코딩 채널 탐색에서는 데이터를 시각화하기 위해 `point` 마크만 독점적으로 사용했습니다. 그러나 `point` 마크 유형은 데이터를 시각적으로 표현하는 데 사용할 수 있는 많은 기하학적 모양 중 하나일 뿐입니다. Altair에는 다음과 같은 여러 내장 마크 유형이 포함되어 있습니다:

- `mark_area()` - 상단 라인과 기준선으로 정의된 채워진 영역.
- `mark_bar()` - 직사각형 막대.
- `mark_circle()` - 채워진 원으로 표시되는 산점도 점.
- `mark_line()` - 연결된 선분.
- `mark_point()` - 모양을 구성할 수 있는 산점도 점.
- `mark_rect()` - 히트맵에 유용한 채워진 직사각형.
- `mark_rule()` - 축을 가로지르는 수직 또는 수평선.
- `mark_square()` - 채워진 사각형으로 표시되는 산점도 점.
- `mark_text()` - 텍스트로 표현되는 산점도 점.
- `mark_tick()` - 수직 또는 수평 틱 마크.

전체 목록과 예시 링크는 [Altair 마크 문서](https://altair-viz.github.io/user_guide/marks/index.html)를 참조하세요. 다음으로 통계 그래픽에 가장 일반적으로 사용되는 몇 가지 마크 유형을 단계별로 살펴보겠습니다.

### 점 마크 (Point Marks)

`point` 마크 유형은 *산점도(scatter plots)* 및 *도트 플롯(dot plots)*에서와 같이 특정 지점을 전달합니다. `x` 및 `y` 인코딩 채널(2D 점 위치 지정) 외에도 점 마크는 `color`, `size` 및 `shape` 인코딩을 사용하여 추가 데이터 필드를 전달할 수 있습니다.

아래는 `fertility`의 도트 플롯이며, `cluster` 필드가 `y` 및 `shape` 채널을 모두 사용하여 중복 인코딩되었습니다.



In [23]:
alt.Chart(data2000).mark_point().encode(
    alt.X('fertility:Q'),
    alt.Y('cluster:N'),
    alt.Shape('cluster:N')
)

alt.Chart(...)

인코딩 채널 외에도 `mark_*()` 메서드에 값을 제공하여 마크를 스타일링할 수 있습니다.

예를 들어, 점 마크는 기본적으로 테두리 선으로 그려지지만, 대신 `filled` 모양을 사용하도록 지정할 수 있습니다. 마찬가지로 기본 `size`를 설정하여 점 마크의 총 픽셀 면적을 설정할 수 있습니다.


In [24]:
alt.Chart(data2000).mark_point(filled=True, size=100).encode(
    alt.X('fertility:Q'),
    alt.Y('cluster:N'),
    alt.Shape('cluster:N')
)

alt.Chart(...)

### 원 마크 (Circle Marks)

`circle` 마크 유형은 채워진 원으로 그려지는 `point` 마크의 편리한 약칭입니다.

In [25]:
alt.Chart(data2000).mark_circle(size=100).encode(
    alt.X('fertility:Q'),
    alt.Y('cluster:N'),
    alt.Shape('cluster:N')
)

alt.Chart(...)

### 사각형 마크 (Square Marks)

`square` 마크 유형은 채워진 사각형으로 그려지는 `point` 마크의 편리한 약칭입니다.

In [26]:
alt.Chart(data2000).mark_square(size=100).encode(
    alt.X('fertility:Q'),
    alt.Y('cluster:N'),
    alt.Shape('cluster:N')
)

alt.Chart(...)

### 틱 마크 (Tick Marks)

`tick` 마크 유형은 짧은 선분 또는 "틱"을 사용하여 데이터 포인트를 전달합니다. 이는 겹침을 최소화하면서 단일 차원을 따라 값을 비교하는 데 특히 유용합니다. 틱 마크로 그려진 *도트 플롯*을 때때로 *스트립 플롯(strip plot)*이라고도 합니다.

In [27]:
alt.Chart(data2000).mark_tick().encode(
    alt.X('fertility:Q'),
    alt.Y('cluster:N'),
    alt.Shape('cluster:N')
)

alt.Chart(...)

### 막대 마크 (Bar Marks)

`bar` 마크 유형은 위치, 너비 및 높이가 있는 직사각형을 그립니다.

아래 플롯은 각 국가의 인구(`pop`)를 나타내는 간단한 막대 차트입니다.

In [28]:
alt.Chart(data2000).mark_bar().encode(
    alt.X('country:N'),
    alt.Y('pop:Q')
)

alt.Chart(...)

막대 너비는 기본 크기로 설정됩니다. 막대 너비를 조정하는 방법은 이 노트북의 뒷부분에서 논의하겠습니다. (나중에 나올 노트북에서 축, 스케일 및 범례 구성에 대해 더 자세히 살펴볼 것입니다.)

막대는 누적될 수도 있습니다. `x` 인코딩을 `cluster` 필드를 사용하도록 변경하고, `color` 채널을 사용하여 `country`를 인코딩해 보겠습니다. 또한 모든 국가에 대한 색상이 있으면 범례가 매우 길어지므로 범례를 비활성화하고 국가 이름에 툴팁을 사용하겠습니다.

In [29]:
alt.Chart(data2000).mark_bar().encode(
    alt.X('cluster:N'),
    alt.Y('pop:Q'),
    alt.Color('country:N', legend=None),
    alt.Tooltip('country:N')
)

alt.Chart(...)

위의 차트에서 `color` 인코딩 채널을 사용하면 Altair / Vega-Lite가 자동으로 막대 마크를 누적합니다. 그렇지 않으면 막대가 서로 겹쳐서 그려질 것입니다! `y` 인코딩 채널에 `stack=None` 매개변수를 추가하여 누적을 적용하지 않으면 어떻게 되는지 확인해 보세요...

위의 예제는 0 기준선에서 막대 차트를 만들고, `y` 채널은 막대의 0이 아닌 값(또는 높이)만 인코딩합니다. 그러나 막대 마크를 사용하면 시작점과 끝점을 지정하여 범위를 전달할 수도 있습니다.

아래 차트는 `x`(시작점) 및 `x2`(끝점) 채널을 사용하여 각 지역 클러스터 내의 기대 수명 범위를 보여줍니다. 아래에서는 `min` 및 `max` 집계 함수를 사용하여 범위의 끝점을 결정합니다. 다음 노트북에서 집계에 대해 더 자세히 다룰 것입니다!

또는 `x` 및 `width`를 사용하여 시작점과 오프셋을 제공할 수 있습니다 (`x2 = x + width`).

In [30]:
alt.Chart(data2000).mark_bar().encode(
    alt.X('min(life_expect):Q'),
    alt.X2('max(life_expect):Q'),
    alt.Y('cluster:N')
)

alt.Chart(...)

### 선 마크 (Line Marks)

`line` 마크 유형은 플로팅된 점을 선분으로 연결합니다. 예를 들어 선의 기울기가 변화율에 대한 정보를 전달하도록 합니다.

전체 글로벌 개발 데이터 프레임을 사용하여 연도별 국가당 출산율 선 차트를 그려 보겠습니다. 다시 한번 범례를 숨기고 대신 툴팁을 사용하겠습니다.


In [31]:
alt.Chart(data).mark_line().encode(
    alt.X('year:O'),
    alt.Y('fertility:Q'),
    alt.Color('country:N', legend=None),
    alt.Tooltip('country:N')
).properties(
    width=400
)

alt.Chart(...)

국가별로 흥미로운 변화를 볼 수 있지만, 전반적으로 시간이 지남에 따라 가구당 자녀 수가 줄어드는 추세를 확인할 수 있습니다. 또한 사용자 정의 너비를 400 픽셀로 설정했습니다. *너비를 변경(또는 제거)해보고 어떤 일이 일어나는지 확인해 보세요!*

플롯을 사용자 정의하기 위해 일부 기본 마크 매개변수를 변경해 보겠습니다. `strokeWidth`를 설정하여 선의 두께를 결정하고 `opacity`를 설정하여 약간의 투명도를 추가할 수 있습니다. 기본적으로 `line` 마크는 직선 세그먼트를 사용하여 데이터 포인트를 연결합니다. 어떤 경우에는 선을 부드럽게 만들고 싶을 수도 있습니다. `interpolate` 마크 매개변수를 설정하여 데이터 포인트를 연결하는 데 사용되는 보간법을 조정할 수 있습니다. 보간의 결과로 의도치 않게 "가짜" 최솟값이나 최댓값이 생성되지 않도록 보장하면서 부드러운 선을 제공하는 `'monotone'` 보간법을 사용해 보겠습니다.

In [32]:
alt.Chart(data).mark_line(
    strokeWidth=3,
    opacity=0.5,
    interpolate='monotone'
).encode(
    alt.X('year:O'),
    alt.Y('fertility:Q'),
    alt.Color('country:N', legend=None),
    alt.Tooltip('country:N')
).properties(
    width=400
)

alt.Chart(...)

`line` 마크는 선의 기울기를 사용하여 두 비교 지점 간의 값 변화를 강조하는 차트인 *경사 그래프(slope graphs)*를 만드는 데에도 사용할 수 있습니다.

아래에서는 전체 데이터 세트의 최소 및 최대 연도인 1955년과 2005년의 각 국가 인구를 비교하는 경사 그래프를 만들어 보겠습니다. 먼저 해당 연도로 필터링된 새 Pandas 데이터 프레임을 만든 다음 Altair를 사용하여 경사 그래프를 만듭니다.

기본적으로 Altair는 연도를 서로 가깝게 배치합니다. x축을 따라 연도를 더 넓게 배치하기 위해 아래 주석에 표시된 대로 차트 너비를 따라 이산적인 단계의 크기(픽셀 단위)를 지정할 수 있습니다. 아래의 너비 `step` 값을 조정해보고 차트가 어떻게 변하는지 확인해 보세요.

In [33]:
dataTime = data.loc[(data['year'] == 1955) | (data['year'] == 2005)]

alt.Chart(dataTime).mark_line(opacity=0.5).encode(
    alt.X('year:O'),
    alt.Y('pop:Q'),
    alt.Color('country:N', legend=None),
    alt.Tooltip('country:N')
).properties(
    width={"step": 50} # step 매개변수 조정
)

alt.Chart(...)

### 영역 마크 (Area Marks)

`area` 마크 유형은 `line` 및 `bar` 마크의 측면을 결합합니다. 데이터 포인트 간의 연결(기울기)을 시각화할 뿐만 아니라 한쪽 가장자리가 기본적으로 0 값 기준선이 되는 채워진 영역을 보여줍니다.

아래 차트는 미국(United States)의 시간에 따른 출산율 변화를 보여주는 영역 차트입니다:

In [34]:
dataUS = data.loc[data['country'] == 'United States']

alt.Chart(dataUS).mark_area().encode(
    alt.X('year:O'),
    alt.Y('fertility:Q')
)

alt.Chart(...)

`line` 마크와 마찬가지로 `area` 마크도 `interpolate` 매개변수를 지원합니다.

In [35]:
alt.Chart(dataUS).mark_area(interpolate='monotone').encode(
    alt.X('year:O'),
    alt.Y('fertility:Q')
)

alt.Chart(...)

`bar` 마크와 마찬가지로 `area` 마크도 누적(stacking)을 지원합니다. 여기서는 북미 3개 국가의 데이터로 새 데이터 프레임을 만든 다음, `area` 마크와 국가별로 누적하기 위한 `color` 인코딩 채널을 사용하여 플로팅합니다.

In [36]:
dataNA = data.loc[
    (data['country'] == 'United States') |
    (data['country'] == 'Canada') |
    (data['country'] == 'Mexico')
]

alt.Chart(dataNA).mark_area().encode(
    alt.X('year:O'),
    alt.Y('pop:Q'),
    alt.Color('country:N')
)

alt.Chart(...)

기본적으로 누적은 0 기준선을 기준으로 수행됩니다. 그러나 다른 `stack` 옵션을 사용할 수 있습니다:

* `center` - 차트 중앙의 기준선을 기준으로 누적하여 *스트림그래프(streamgraph)* 시각화를 생성합니다.
* `normalize` - 각 누적 지점에서 합산된 데이터를 100%로 정규화하여 백분율 비교가 가능하게 합니다.

아래에서는 `y` 인코딩의 `stack` 속성을 `center`로 설정하여 차트를 수정합니다. 대신 `normalize`로 설정하면 어떻게 될까요?

In [37]:
alt.Chart(dataNA).mark_area().encode(
    alt.X('year:O'),
    alt.Y('pop:Q', stack='center'),
    alt.Color('country:N')
)

alt.Chart(...)

누적을 완전히 비활성화하려면 `stack` 속성을 `None`으로 설정하세요. 겹치는 영역을 볼 수 있도록 기본 마크 매개변수로 `opacity`를 추가할 수도 있습니다!

In [38]:
alt.Chart(dataNA).mark_area(opacity=0.5).encode(
    alt.X('year:O'),
    alt.Y('pop:Q', stack=None),
    alt.Color('country:N')
)

alt.Chart(...)

`area` 마크 유형은 상단 및 하단 시리즈가 모두 데이터 필드에 의해 결정되는 데이터 구동 기준선도 지원합니다. `bar` 마크와 마찬가지로 `x` 및 `x2`(또는 `y` 및 `y2`) 채널을 사용하여 영역 마크의 끝점을 제공할 수 있습니다.

아래 차트는 북미 국가들의 연도별 최소 및 최대 출산율 범위를 시각화합니다:

In [39]:
alt.Chart(dataNA).mark_area().encode(
    alt.X('year:O'),
    alt.Y('min(fertility):Q'),
    alt.Y2('max(fertility):Q')
).properties(
    width={"step": 40}
)

alt.Chart(...)

1995년에는 4 미만부터 7 미만까지 더 넓은 범위의 값을 볼 수 있습니다. 2005년이 되면 전체 출산율 값과 변동성이 모두 감소하여 가구당 자녀 수가 2명 정도로 집중됩니다.

위의 모든 `area` 마크 예제는 수직 방향 영역을 사용합니다. 그러나 Altair와 Vega-Lite는 수평 영역도 지원합니다. `x` 및 `y` 채널을 바꾸기만 하면 위의 차트를 전치(transpose)할 수 있습니다.

In [40]:
alt.Chart(dataNA).mark_area().encode(
    alt.Y('year:O'),
    alt.X('min(fertility):Q'),
    alt.X2('max(fertility):Q')
).properties(
    width={"step": 40}
)

alt.Chart(...)

## 요약 (Summary)

데이터 유형, 인코딩 채널 및 그래픽 마크에 대한 탐색을 마쳤습니다! 이제 인코딩, 마크 유형 및 마크 매개변수의 공간을 더 탐구할 수 있는 충분한 준비가 되었을 것입니다. 여기서 생략한 기능을 포함한 포괄적인 참조는 Altair [마크](https://altair-viz.github.io/user_guide/marks/index.html) 및 [인코딩](https://altair-viz.github.io/user_guide/encodings/index.html) 문서를 참조하세요.

다음 모듈에서는 데이터를 요약하거나 새로운 파생 필드를 시각화하는 차트를 만들기 위한 데이터 변환 사용에 대해 살펴보겠습니다. 나중에 나올 모듈에서는 스케일, 축 및 범례를 수정하여 차트를 더 자세히 사용자 정의하는 방법을 살펴볼 것입니다.

시각적 인코딩에 대해 더 자세히 알고 싶으신가요?

<img title="Bertin's Taxonomy of Visual Encoding Channels" src="https://cdn-images-1.medium.com/max/2000/1*jsb78Rr2cDy6zrE7j2IKig.png" style="max-width: 650px;"><br/>

<small>Bertin's taxonomy of visual encodings from <a href="https://books.google.com/books/about/Semiology_of_Graphics.html?id=X5caQwAACAAJ"><em>Sémiologie Graphique</em></a>, as adapted by <a href="https://bost.ocks.org/mike/">Mike Bostock</a>.</small>

- 마크, 시각적 인코딩 및 뒷받침하는 데이터 유형에 대한 체계적인 연구는 [Jacques Bertin](https://en.wikipedia.org/wiki/Jacques_Bertin)이 1967년의 선구적인 저서 [*Sémiologie Graphique (The Semiology of Graphics)*](https://books.google.com/books/about/Semiology_of_Graphics.html?id=X5caQwAACAAJ)에서 시작했습니다. 위의 이미지는 위치, 크기, 값(밝기), 질감, 색상(색조), 방향 및 모양 채널과 함께 Bertin이 권장하는 데이터 유형을 보여줍니다.
- 데이터 유형, 마크 및 채널의 프레임워크는 1986년 [Mackinlay의 APT (A Presentation Tool)](https://scholar.google.com/scholar?cluster=10191273548472217907)를 시작으로 [Voyager](http://idl.cs.washington.edu/papers/voyager/) 및 [Draco](http://idl.cs.washington.edu/papers/draco/)와 같은 최근 시스템에 이르기까지 *자동화된* 시각화 디자인 도구의 가이드 역할도 합니다.
- 명목형, 서수형, 간격 및 비율 유형의 식별은 적어도 1947년 S. S. Steven의 기사 [*On the theory of scales of measurement*](https://scholar.google.com/scholar?cluster=14356809180080326415)까지 거슬러 올라갑니다.